# Semana 12 — Issues, Boards, Pull Requests e Quality Gates
## Rastreabilidade do requisito ao merge protegido

Na Semana 10, implementamos testes de integração e E2E. Agora eles deixam de ser uma execução opcional: cada alteração nasce em uma issue, percorre o board e só pode entrar na branch protegida depois de revisão e verificações automáticas.

> **Pergunta central:** como provar por que uma mudança existe, quem a revisou e quais critérios foram executados antes do merge?

### Objetivos de aprendizagem

- escrever issues com valor e critérios verificáveis;
- conectar issue, item do Project, branch, commits e Pull Request;
- aplicar GitFlow sem perder rastreabilidade;
- criar workflow de CI para lint, unitários, integração, E2E e cobertura;
- distinguir workflow executado de Quality Gate obrigatório;
- interpretar falhas e atualizar a PR;
- definir regras de proteção coerentes com o risco.

![Da issue ao merge protegido](img/semana_12_issue_pr_quality_gate.png)

## 1. Cadeia de evidências

```mermaid
flowchart LR
 I[Issue + critérios] --> B[Card no board]
 I --> F[feature/numero-descricao]
 F --> C[Commits pequenos]
 C --> PR[Pull Request]
 PR --> R[Revisão]
 PR --> CI[Quality Gate]
 CI -->|passa| M[Merge em develop/main]
 CI -->|falha| F
 M --> I
```

Uma issue não é apenas lembrete; ela registra problema, valor, escopo e aceite. A PR não é apenas botão de merge; ela reúne discussão, diff e evidências.

## 2. Issue orientada a resultado

```markdown
## Problema
Alertas duplicados são exibidos quando uma mensagem QoS 1 é reenviada.

## História
Como operador, quero visualizar cada evento uma única vez para não reconhecer o mesmo alerta repetidamente.

## Critérios de aceitação
- [ ] Mensagens com o mesmo event_id produzem um único alerta.
- [ ] Duplicatas são registradas sem gerar efeito de negócio.
- [ ] Há teste de integração para reenvio.
- [ ] README descreve a decisão.

## Fora do escopo
Deduplicação distribuída entre regiões.
```

Critérios devem ser observáveis. “Melhorar o sistema” não é verificável.

## 3. Board e limites de trabalho

Colunas possíveis: `Backlog → Pronto → Em andamento → Em revisão → Concluído`. Campos úteis: prioridade, responsável, tamanho e iteração.

Um limite de WIP reduz trabalho iniciado e não terminado. O board representa estado do trabalho; Git representa estado do código. A automação pode mover o item quando a PR abre ou fecha, mas o significado precisa ser definido pela equipe.

## 4. GitFlow com rastreabilidade

```bash
git switch develop
git pull --ff-only
git switch -c feature/123-deduplicar-alertas

git add src tests
git commit -m "feat(alertas): ignorar event_id já processado"
git push -u origin feature/123-deduplicar-alertas
```

O número conecta branch à issue. Na PR, `Closes #123` fecha a issue quando a PR entra na branch padrão; quando o destino é `develop`, confirme o comportamento desejado e use links explícitos para não encerrar prematuramente.

## 5. Pull Request como unidade de integração

Um bom corpo de PR contém:

- problema e issue relacionada;
- o que mudou e o que não mudou;
- estratégia de teste;
- evidências relevantes;
- riscos e rollback;
- checklist de aceite.

Prefira PRs pequenas e coesas. Revisão efetiva avalia comportamento, segurança, manutenção, testes e documentação — não apenas estilo.

## 6. Workflow: executar as verificações

```yaml
name: quality-gate
on:
  pull_request:
    branches: [develop, main]

permissions:
  contents: read

jobs:
  quality:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v5
        with:
          python-version: "3.12"
          cache: pip
      - run: pip install -r requirements.txt
      - run: pip install ruff pytest pytest-cov
      - name: Lint
        run: ruff check .
      - name: Unitários
        run: pytest tests/unit -q
      - name: Integração
        run: pytest tests/integration -q
      - name: E2E
        run: pytest tests/e2e -q
      - name: Cobertura
        run: pytest --cov=src --cov-report=term-missing --cov-fail-under=80
```

Fixar versões maiores oficiais é comum em exemplos; políticas mais rigorosas podem fixar SHA para reduzir risco de cadeia de suprimentos.

## 7. Quality Gate é uma política, não apenas YAML

O workflow gera checks. Para bloquear merge, a branch deve exigir esses checks por ruleset/branch protection. Configure, conforme disponibilidade do repositório:

- Pull Request obrigatório;
- uma aprovação;
- conversas resolvidas;
- check `quality` obrigatório;
- branch atualizada antes do merge;
- proibição de force push;
- aplicação da regra a administradores quando apropriado.

O check obrigatório precisa existir no último commit da PR. Filtros que impedem o workflow de executar podem deixar o check pendente.

## 8. Separação de jobs e dependências

```mermaid
flowchart LR
 L[lint] --> Q[quality-gate]
 U[unit] --> Q
 I[integration] --> Q
 E[e2e] --> Q
 C[coverage] --> Q
 Q -->|sucesso| M[merge liberado]
```

Jobs independentes dão feedback paralelo e diagnóstico claro. Um job agregador pode depender de todos. E2E costuma ser mais lento e exigir serviços; use `services`, Compose ou ambiente efêmero sem credenciais de produção.

## 9. Simulação de política

O modelo abaixo deixa explícito que aprovação humana e checks automáticos são condições distintas.

In [ ]:
def avaliar_pr(aprovacoes, conversas_abertas, checks):
    falhas=[]
    if aprovacoes < 1: falhas.append("falta aprovação")
    if conversas_abertas: falhas.append("há conversas não resolvidas")
    falhas += [f"check {nome} falhou" for nome,status in checks.items() if status != "success"]
    return {"merge_liberado":not falhas,"motivos":falhas}

cenarios=[
 (1,False,{"lint":"success","integration":"success","e2e":"success"}),
 (1,False,{"lint":"success","integration":"failure","e2e":"success"}),
 (0,True,{"lint":"success","integration":"success","e2e":"success"}),
]
for c in cenarios: print(avaliar_pr(*c))

## 10. Falha útil e ciclo de correção

1. check falha e bloqueia a PR;
2. desenvolvedor lê o primeiro erro relevante;
3. reproduz localmente com o mesmo comando;
4. corrige a causa, não o teste sem justificativa;
5. cria commit adicional e envia;
6. workflow roda no novo SHA;
7. revisão confirma a solução.

Não use `continue-on-error` em um gate obrigatório para ocultar falhas. Teste intermitente precisa ser investigado, não repetido até passar.

## 11. Exercícios práticos

### Exercício 1 — Issue e board
Crie uma issue real do projeto com história, quatro critérios verificáveis e fora do escopo. Adicione ao board e defina prioridade.

### Exercício 2 — Rastreabilidade GitFlow
Crie branch contendo o número da issue, faça dois commits atômicos e abra PR vinculada. Inclua evidências dos testes.

### Exercício 3 — Primeiro Quality Gate
Implemente workflow com Ruff, unitários, integração, E2E e cobertura mínima. Garanta execução em PR para `develop`.

### Exercício 4 — Falha controlada
Introduza temporariamente uma violação de lint e uma regressão. Capture o bloqueio, corrija e prove a nova execução verde.

### Exercício 5 — Proteção
Configure ruleset/branch protection com PR e check obrigatório. Tente merge com check falhando e documente o resultado.

### Exercício 6 — Auditoria
Escolha uma funcionalidade e monte a cadeia: issue → card → branch → commits → PR → checks → merge.

**Entregável:** links/evidências da cadeia, workflow YAML comentado e reflexão de até 300 palavras.

## Bibliografia

- GITHUB. **Linking a pull request to an issue**. <https://docs.github.com/en/issues/tracking-your-work-with-issues/using-issues/linking-a-pull-request-to-an-issue>.
- GITHUB. **About Projects**. <https://docs.github.com/en/issues/planning-and-tracking-with-projects/learning-about-projects/about-projects>.
- GITHUB. **Building and testing Python**. <https://docs.github.com/en/actions/tutorials/build-and-test-code/python>.
- GITHUB. **Status checks**. <https://docs.github.com/en/pull-requests/collaborating-with-pull-requests/collaborating-on-repositories-with-code-quality-features/about-status-checks>.
- GITHUB. **About protected branches**. <https://docs.github.com/en/repositories/configuring-branches-and-merges-in-your-repository/managing-protected-branches/about-protected-branches>.
- HUMBLE, J.; FARLEY, D. *Continuous Delivery*. Addison-Wesley, 2010.
- FORSGREN, N.; HUMBLE, J.; KIM, G. *Accelerate*. IT Revolution, 2018.